# 00 — Environment and source manifest

**Objective.** Mount Drive, resolve the real source commit, verify the four frozen CSV hashes, and create deterministic seed/runtime manifests.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted data blocks: source files only; no cohort labels or model results.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("00", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import os
os.environ["CRUX_PROFILE"] = "full"

In [ ]:
import json, platform, subprocess, sys
import pandas as pd
from cruxvc.hashing import sha256_file
from cruxvc.io import write_json, write_table
from cruxvc.source import ensure_git_source, locate_source_csvs, verify_source_files, copy_source_files, write_resolved_manifest
from cruxvc.workflow import seed_registry

source_cfg = CFG["source"]
expected_files = list(source_cfg["expected_files"])
source_checkout = P.raw / "crunchbase-october-2013-source"
stable_raw = P.raw / "crunchbase_october_2013"

In [ ]:
checkout = ensure_git_source(
    source_cfg["repository_url"],
    source_checkout,
    branch=source_cfg.get("branch", "master"),
    update=os.environ.get("CRUX_UPDATE_SOURCE", "0") == "1",
)
located = locate_source_csvs(source_checkout, expected_files)
verification = verify_source_files(
    located,
    source_cfg["expected_files"],
    strict=bool(CFG["execution"]["strict_expected_source_hashes"]),
)
copied = copy_source_files(located, stable_raw)
resolved_manifest = write_resolved_manifest(
    P.protocol / "source_manifest.json",
    checkout,
    verification,
    extra={
        "legacy_unverified_commit": source_cfg.get("legacy_unverified_commit"),
        "administrative_cutoff": source_cfg["administrative_cutoff"],
        "note": "checked_out_commit was resolved by execution; no hard-coded commit was trusted",
    },
)
print(json.dumps({"commit": checkout["commit"], "hashes_match": all(v["match"] for v in verification.values())}, indent=2))

In [ ]:
full = CFG["compute_profiles"]["full"]
seeds = seed_registry(
    int(CFG["execution"]["random_seed"]),
    {
        "model_selection": 256,
        "bootstrap_refit": int(full["bootstrap_refits"]),
        "background": int(full["background_sets"]),
        "approximation": int(full["approximation_seeds"]),
        "inference_bootstrap": 32,
        "simulation": 32,
        "control": 32,
    },
)
seed_path = write_table(seeds, P.protocol / "seed_registry.csv")
runtime_path = write_json(
    {
        "python": sys.version,
        "platform": platform.platform(),
        "repository_root": str(P.root),
        "compute_profile": PROFILE,
        "requirements_sha256": sha256_file(P.root / "requirements.txt"),
        "project_config_sha256": sha256_file(P.config / "project.yaml"),
        "author_email": CFG["project"]["author_email"],
    },
    P.protocol / "runtime_manifest.json",
)

In [ ]:
outputs = [resolved_manifest, seed_path, runtime_path, *copied.values()]
CTX.recorder.complete(outputs, extra={"resolved_source_commit": checkout["commit"]})
print("Stage 00 completed. Raw source files are hash-verified and immutable by convention.")